# Airbnb Price Prediction — Sklearn Pipeline

Цель: создание воспроизводимого ML-пайплайна для
прогнозирования стоимости аренды жилья Airbnb.

В рамках данного этапа:
- загрузим данные из S3;
- выполним очистку данных и разделение на train/test;
- объединим предобработку и модель в sklearn Pipeline;
- оценим качество модели;
- сохраним обученный Pipeline в model.pkl;
- проверим корректность предсказаний после загрузки.

In [2]:
import os
import pickle
import boto3
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

RANDOM_STATE = 42

In [3]:
load_dotenv()

S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/listings.csv"

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    region_name="ru-central1"
)

response = s3.get_object(
    Bucket=S3_BUCKET,
    Key=S3_KEY
)

with response["Body"] as body:
    df = pd.read_csv(
        body,
        dtype={"listing_id": str}
    )

print("Размер датасета:", df.shape)
display(df.head())

/Users/polina.sokol/Projects/airbnb-mlops/.venv/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Размер датасета: (20151, 13)


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


In [4]:
df = df.drop_duplicates().copy()

df = df[
    df["price"].notna()
    & np.isfinite(df["price"])
    & (df["price"] > 0)
].reset_index(drop=True)

print("Размер после очистки:", df.shape)

Размер после очистки: (19191, 13)


In [5]:
numeric_features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights"
]

categorical_features = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

features = numeric_features + categorical_features

X = df[features].copy()
y = df["price"].copy()
groups = df["listing_id"].copy()

print("Количество признаков:", len(features))

Количество признаков: 10


In [6]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_ids = set(groups.iloc[train_idx])
test_ids = set(groups.iloc[test_idx])

assert train_ids.isdisjoint(test_ids)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Пересечение ID:", len(train_ids & test_ids))

Train: (15352, 10)
Test: (3839, 10)
Пересечение ID: 0


## Data preprocessing

Для обработки различных типов признаков используется
ColumnTransformer.

Числовые пропуски заполняются медианными значениями,
категориальные — наиболее частыми значениями.

Категориальные признаки преобразуются с помощью
OneHotEncoder.

Все обучаемые преобразования настраиваются исключительно
на тренировочной выборке, что предотвращает утечку данных.

In [7]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['latitude', 'longitude', 'accommodates',
                                  'bathrooms', 'bedrooms', 'beds',
                                  'minimum_nights']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['neighbourhood_cleansed', 'property_type',
                                  'room_type'])])

In [8]:
regressor = TransformedTargetRegressor(
    regressor=RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", regressor)
    ]
)

pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['latitude', 'longitude',
                                                   'accommodates', 'bathrooms',
                                                   'bedrooms', 'beds',
                                                   'minimum_nights']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['neighbourhood_cleansed',
                                                   'property_type',
                                                   'room_type'])])),
                ('model',
                 TransformedTargetRegressor(func=<ufunc 'log1p'>,
                                            inverse_func=<ufunc 'expm1'>,
                                            regressor=RandomForestRegressor(n_estimators=200,
                                                                            n_jobs=-1,
                                                                            random_state=42)))])

In [9]:
pipeline.fit(X_train, y_train)

print("Pipeline успешно обучен")

Pipeline успешно обучен


In [10]:
y_pred = pipeline.predict(X_test)

results = pd.DataFrame({
    "MAE": [
        mean_absolute_error(y_test, y_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred))
    ],
    "R2": [
        r2_score(y_test, y_pred)
    ]
}, index=["Random Forest Pipeline"])

display(results.round(3))

,MAE,RMSE,R2
Random Forest Pipeline,96.738,382.317,0.171


## Model serialization

Обученный Pipeline сохраняется в формате pickle.

Сериализованный объект содержит все обученные
преобразования признаков и модель Random Forest.

Это позволяет выполнять предсказания на новых данных
без повторного обучения и ручной предобработки.

In [11]:
MODEL_PATH = "model.pkl"

with open(MODEL_PATH, "wb") as file:
    pickle.dump(pipeline, file)

print("Модель сохранена:", MODEL_PATH)

print(
    "Размер файла, MB:",
    round(os.path.getsize(MODEL_PATH) / 1024**2, 2)
)

Модель сохранена: model.pkl
Размер файла, MB: 198.96


In [12]:
with open(MODEL_PATH, "rb") as file:
    loaded_pipeline = pickle.load(file)

loaded_predictions = loaded_pipeline.predict(X_test)

print("Модель успешно загружена")
print("Количество предсказаний:", len(loaded_predictions))

Модель успешно загружена
Количество предсказаний: 3839


In [13]:
np.testing.assert_allclose(
    y_pred,
    loaded_predictions,
    rtol=1e-10,
    atol=1e-10
)

print("Предсказания до и после сохранения совпадают")

comparison = pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "original_prediction": y_pred,
    "loaded_prediction": loaded_predictions
})

display(comparison.head(10).round(2))

Предсказания до и после сохранения совпадают


,actual_price,original_prediction,loaded_prediction
0,126.00,138.82,138.82
1,53.00,55.46,55.46
2,246.50,314.42,314.42
3,366.00,293.04,293.04
4,427.70,492.61,492.61
5,95.00,153.73,153.73
6,160.00,102.30,102.30
7,82.00,78.20,78.20
8,480.00,176.81,176.81
9,300.71,257.59,257.59


In [14]:
new_listing = pd.DataFrame([{
    "latitude": 51.5074,
    "longitude": -0.1278,
    "accommodates": 4,
    "bathrooms": 1.0,
    "bedrooms": 2,
    "beds": 2,
    "minimum_nights": 2,
    "neighbourhood_cleansed": "Westminster",
    "property_type": "Entire rental unit",
    "room_type": "Entire home/apt"
}])

prediction = loaded_pipeline.predict(new_listing)

display(new_listing)

print(f"Предсказанная стоимость: {prediction[0]:.2f}")

,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type
0,51.5074,-0.1278,4,1.0,2,2,2,Westminster,Entire rental unit,Entire home/apt


Предсказанная стоимость: 511.76


## Conclusion

В рамках данного этапа реализован единый sklearn Pipeline,
объединяющий предобработку числовых и категориальных
признаков, логарифмическое преобразование целевой
переменной и модель Random Forest.

На тестовой выборке получены следующие метрики:
- MAE: 96.738
- RMSE: 382.317
- R²: 0.171

По сравнению с baseline средняя абсолютная ошибка
снизилась на 35.3%.

Обученный Pipeline сериализован в файл model.pkl.
Проверено, что предсказания до и после сериализации
совпадают на всей тестовой выборке.

Также успешно выполнено предсказание стоимости
для нового объявления без дополнительного обучения
и ручной предобработки данных.

In [15]:
MODEL_S3_KEY = "models/model.pkl"

s3.upload_file(
    MODEL_PATH,
    S3_BUCKET,
    MODEL_S3_KEY
)

print(f"Модель загружена в s3://{S3_BUCKET}/{MODEL_S3_KEY}")

Модель загружена в s3://mlops26-course-bucket/models/model.pkl


In [16]:
response = s3.head_object(
    Bucket=S3_BUCKET,
    Key=MODEL_S3_KEY
)

print("Модель найдена в S3")
print(
    "Размер модели, МБ:",
    round(response["ContentLength"] / 1024**2, 2)
)

Модель найдена в S3
Размер модели, МБ: 198.96
